site: https://brasileirao-2026-simulator.vercel.app/


In [ ]:
# ============================================================
# CÉLULA 1 — INSTALAÇÃO
# ============================================================
!pip install -q pandas numpy openpyxl xlsxwriter

print("Intalações prontas!")

In [ ]:
# ============================================================
# CÉLULA 2 — IMPORTS E CONFIGURAÇÃO
# ============================================================
import pandas as pd
import numpy as np
import io
from google.colab import files

# ── Parâmetros do simulador (altere apenas aqui se necessário)
NUM_SIMULACOES   = 10_000
VANTAGEM_MANDANTE = 1.15   # multiplicador de gols para o mandante
FATOR_FASE_MEDIA  = 1.0    # média do Gaussian Noise
FATOR_FASE_DESVIO = 0.20   # desvio padrão — estica a tabela
MU                = 1.3    # média de gols por time por jogo no Brasileirão
LAMBDA_MIN        = 0.3    # clipping inferior
LAMBDA_MAX        = 4.0    # clipping superior

# ── Mapeamento de nomes (mesmo padrão do coletor)
MAPA_TIMES = {
    'CAM': 'Atlético Mineiro',  'PAL': 'Palmeiras',
    'INT': 'Internacional',     'CAP': 'Athletico Paranaense',
    'CFC': 'Coritiba',          'RBB': 'Bragantino',
    'VIT': 'Vitória',           'REM': 'Remo',
    'FLU': 'Fluminense',        'GRE': 'Grêmio',
    'COR': 'Corinthians',       'BAH': 'Bahia',
    'CHA': 'Chapecoense',       'SAN': 'Santos',
    'SAO': 'São Paulo',         'FLA': 'Flamengo',
    'MIR': 'Mirassol',          'VAS': 'Vasco',
    'BOT': 'Botafogo',          'CRU': 'Cruzeiro',
    'Atlético-MG': 'Atlético Mineiro',
    'Athletico-PR': 'Athletico Paranaense',
    'Red Bull Bragantino': 'Bragantino',
    'RB Bragantino': 'Bragantino',
    'Vasco da Gama': 'Vasco',
}

TIMES_SERIE_A = [
    'Atlético Mineiro', 'Athletico Paranaense', 'Bahia', 'Botafogo',
    'Bragantino', 'Chapecoense', 'Corinthians', 'Coritiba', 'Cruzeiro',
    'Flamengo', 'Fluminense', 'Grêmio', 'Internacional', 'Mirassol',
    'Palmeiras', 'Remo', 'Santos', 'São Paulo', 'Vasco', 'Vitória'
]

def normalizar_time(nome):
    return MAPA_TIMES.get(str(nome).strip(), str(nome).strip())

print("Configuração carregada!")
print(f"   Simulações : {NUM_SIMULACOES:,}")
print(f"   Fator Fase : normal({FATOR_FASE_MEDIA}, {FATOR_FASE_DESVIO})")
print(f"   Vant. Casa : {VANTAGEM_MANDANTE}x")

In [ ]:
from google.colab import userdata
# =====================================================================
# CÉLULA 3 — CONFIGURAÇÃO + ELENCOS FM26 DO DRIVE (SEM UPLOAD MANUAL)
# ATE_RODADA:
#   None  = rodada atual (todos os jogos disponíveis)
#   5     = histórico: simula como se fosse após a Rodada 5
# =====================================================================

import io
import requests
import pandas as pd
from collections import defaultdict
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

SUPABASE_URL        = userdata.get("SUPABASE_URL")
SUPABASE_SECRET_KEY = userdata.get("SUPABASE_SECRET_KEY")

ATE_RODADA = 34 # ← mude aqui a cada rodada nova para rodar histórico ou deixe None que roda todas.

headers_sb = {
    "apikey"       : SUPABASE_SECRET_KEY,
    "Authorization": f"Bearer {SUPABASE_SECRET_KEY}",
    "Content-Type" : "application/json",
    "Prefer"       : "resolution=merge-duplicates",
}

# ── Carrega elencos FM26 direto do Drive (sem upload manual) ─────────
FILE_ID_ELENCOS = "COLE_AQUI_O_ID_DO_ARQUIVO_NO_DRIVE"

auth.authenticate_user()
drive_service = build('drive', 'v3', cache_discovery=False)
request = drive_service.files().get_media(fileId=FILE_ID_ELENCOS)
fh = io.BytesIO()
downloader = MediaIoBaseDownload(fh, request)
done = False
while not done:
    _, done = downloader.next_chunk()
fh.seek(0)
df_fm = pd.read_excel(fh)
df_fm["Time"] = df_fm["Time"].apply(normalizar_time)
print(f"{len(df_fm)} jogadores | {df_fm['Time'].nunique()} times (carregado do Drive)")
print(f"\nATE_RODADA = {ATE_RODADA}")
print("Os jogos e a classificação oficial serão processados na Célula 4.")

In [ ]:
# =====================================================================
# CÉLULA 4 — JOGOS SÉRIE A 2026 + FORÇAS + CLASSIFICAÇÃO SUPABASE
# Para histórico: mude ATE_RODADA na Célula 3
# Para rodada atual: deixe ATE_RODADA = None na Célula 3
# Para adicionar nova rodada: crie JOGOS_R18, adicione em TODAS_RODADAS
# =====================================================================

import pandas as pd
import numpy as np
import requests
from collections import defaultdict

# ── Blocos de jogos por rodada ────────────────────────────────────────

JOGOS_R1 = [
    (1,'CAM','PAL',2,2),
    (1,'CFC','RBB',0,1),
    (1,'INT','CAP',0,1),
    (1,'VIT','REM',2,0),
    (1,'FLU','GRE',2,1),
    (1,'CHA','SAN',4,2),
    (1,'COR','BAH',1,2),
    (1,'SAO','FLA',2,1),
    (1,'MIR','VAS',2,1),
    (1,'BOT','CRU',4,0),
]

JOGOS_R2 = [
    (2,'RBB','CAM',1,0),
    (2,'FLA','INT',1,1),
    (2,'REM','MIR',2,2),
    (2,'SAN','SAO',1,1),
    (2,'GRE','BOT',5,3),
    (2,'PAL','VIT',5,1),
    (2,'BAH','FLU',1,1),
    (2,'VAS','CHA',1,1),
    (2,'CRU','CFC',1,2),
    (2,'CAP','COR',0,1),
]

JOGOS_R3 = [
    (3,'VIT','FLA',1,2),
    (3,'CHA','CFC',3,3),
    (3,'MIR','CRU',2,2),
    (3,'CAM','REM',3,3),
    (3,'SAO','GRE',2,0),
    (3,'VAS','BAH',0,1),
    (3,'CAP','SAN',2,1),
    (3,'FLU','BOT',1,0),
    (3,'COR','RBB',2,0),
    (3,'INT','PAL',1,3),
]

JOGOS_R4 = [
    (4,'RBB','CAP',1,1),
    (4,'REM','INT',1,1),
    (4,'CFC','SAO',0,1),
    (4,'CRU','COR',1,1),
    (4,'GRE','CAM',2,1),
    (4,'PAL','FLU',2,1),
    (4,'SAN','VAS',2,1),
    (4,'FLA','MIR',2,0),
    (4,'BOT','VIT',0,0),
    (4,'BAH','CHA',2,0),
]

JOGOS_R5 = [
    (5,'MIR','SAN',2,2),
    (5,'CAM','INT',1,0),
    (5,'BAH','VIT',1,1),
    (5,'COR','CFC',0,2),
    (5,'FLA','CRU',2,0),
    (5,'REM','FLU',0,2),
    (5,'VAS','PAL',2,1),
    (5,'SAO','CHA',2,0),
    (5,'GRE','RBB',1,1),
    (5,'CAP','BOT',4,1),
]

JOGOS_R6 = [
    (6,'VIT','CAM',2,0),
    (6,'BOT','FLA',0,3),
    (6,'FLU','CAP',3,2),
    (6,'INT','BAH',0,1),
    (6,'SAN','COR',1,1),
    (6,'CFC','REM',1,0),
    (6,'PAL','MIR',1,0),
    (6,'RBB','SAO',1,2),
    (6,'CRU','VAS',3,3),
    (6,'CHA','GRE',1,1),
]

JOGOS_R7 = [
    (7,'BAH','RBB',2,0),
    (7,'PAL','BOT',2,1),
    (7,'CAP','CRU',2,1),
    (7,'CAM','SAO',1,0),
    (7,'MIR','CFC',0,1),
    (7,'SAN','INT',1,2),
    (7,'VAS','FLU',3,2),
    (7,'GRE','VIT',2,0),
    (7,'FLA','REM',3,0),
    (7,'CHA','COR',0,0),
]

JOGOS_R8 = [
    (8,'RBB','BOT',1,2),
    (8,'FLU','CAM',1,0),
    (8,'SAO','PAL',0,1),
    (8,'CAP','CFC',2,0),
    (8,'CRU','SAN',0,0),
    (8,'REM','BAH',4,1),
    (8,'VAS','GRE',2,1),
    (8,'INT','CHA',2,0),
    (8,'VIT','MIR',1,0),
    (8,'COR','FLA',1,1),
]

JOGOS_R9 = [
    (9,'BOT','MIR',3,2),
    (9,'INT','SAO',1,1),
    (9,'BAH','CAP',3,0),
    (9,'CRU','VIT',3,0),
    (9,'CFC','VAS',1,1),
    (9,'FLU','COR',3,1),
    (9,'CHA','CAM',0,4),
    (9,'SAN','REM',2,0),
    (9,'RBB','FLA',3,0),
    (9,'PAL','GRE',2,1),
]

JOGOS_R10 = [
    (10,'SAO','CRU',4,1),
    (10,'CFC','FLU',1,1),
    (10,'VAS','BOT',1,2),
    (10,'CHA','VIT',1,1),
    (10,'CAM','CAP',2,1),
    (10,'FLA','SAN',3,1),
    (10,'BAH','PAL',1,2),
    (10,'COR','INT',0,1),
    (10,'MIR','RBB',0,1),
    (10,'GRE','REM',0,0),
]

JOGOS_R11 = [
    (11,'REM','VAS',1,1),
    (11,'VIT','SAO',2,0),
    (11,'MIR','BAH',1,2),
    (11,'SAN','CAM',1,0),
    (11,'INT','GRE',0,0),
    (11,'CAP','CHA',2,0),
    (11,'BOT','CFC',2,2),
    (11,'FLU','FLA',1,2),
    (11,'COR','PAL',0,0),
    (11,'CRU','RBB',2,1),
]

JOGOS_R12 = [
    (12,'CHA','BOT',1,4),
    (12,'VAS','SAO',2,1),
    (12,'VIT','COR',0,0),
    (12,'CRU','GRE',2,0),
    (12,'INT','MIR',1,2),
    (12,'CFC','CAM',2,0),
    (12,'SAN','FLU',2,3),
    (12,'RBB','REM',4,2),
    (12,'PAL','CAP',1,0),
    (12,'FLA','BAH',2,0),
]

JOGOS_R13 = [
    (13,'BAH','SAN',2,2),
    (13,'BOT','INT',2,2),
    (13,'REM','CRU',0,1),
    (13,'SAO','MIR',1,0),
    (13,'COR','VAS',1,0),
    (13,'GRE','CFC',1,0),
    (13,'CAP','VIT',3,1),
    (13,'RBB','PAL',0,1),
    (13,'CAM','FLA',0,4),
    (13,'FLU','CHA',2,1),
]

JOGOS_R14 = [
    (14,'BOT','REM',1,2),
    (14,'PAL','SAN',1,1),
    (14,'VIT','CFC',4,1),
    (14,'CAP','GRE',0,0),
    (14,'CRU','CAM',1,3),
    (14,'FLA','VAS',2,2),
    (14,'CHA','RBB',1,2),
    (14,'INT','FLU',2,0),
    (14,'SAO','BAH',2,2),
    (14,'MIR','COR',2,1),
]

JOGOS_R15 = [
    (15,'CFC','INT',2,2),
    (15,'FLU','VIT',2,2),
    (15,'BAH','CRU',1,2),
    (15,'CAM','BOT',1,1),
    (15,'REM','PAL',1,1),
    (15,'COR','SAO',3,2),
    (15,'MIR','CHA',1,1),
    (15,'SAN','RBB',2,0),
    (15,'GRE','FLA',0,1),
    (15,'VAS','CAP',1,0),
]

JOGOS_R16 = [
    (16,'CAM','MIR',3,1),
    (16,'INT','VAS',4,1),
    (16,'FLU','SAO',2,1),
    (16,'PAL','CRU',1,1),
    (16,'SAN','CFC',0,3),
    (16,'BAH','GRE',1,1),
    (16,'BOT','COR',3,1),
    (16,'CHA','REM',2,3),
    (16,'RBB','VIT',2,0),
    (16,'CAP','FLA',1,1),
]

JOGOS_R17 = [
    (17,'SAO','BOT',1,1),
    (17,'VIT','INT',2,0),
    (17,'GRE','SAN',3,2),
    (17,'MIR','FLU',1,0),
    (17,'FLA','PAL',0,3),
    (17,'CRU','CHA',2,1),
    (17,'REM','CAP',1,2),
    (17,'COR','CAM',1,0),
    (17,'VAS','RBB',0,3),
    (17,'CFC','BAH',3,2),
]

JOGOS_R18 = [
    (18,'CAP','MIR',1,0),
    (18,'FLA','CFC',3,0),
    (18,'BAH','BOT',2,1),
    (18,'GRE','COR',1,3),
    (18,'SAN','VIT',3,1),
    (18,'RBB','INT',3,1),
    (18,'PAL','CHA',1,0),
    (18,'VAS','CAM',0,1),
    (18,'CRU','FLU',1,1),
    (18,'REM','SAO',1,0),
]

JOGOS_R19 = [
    (19,'BOT','SAN',2,1),
    (19,'VIT','VAS',1,0),
    (19,'FLU','RBB',1,1),
    (19,'MIR','GRE',2,1),
    (19,'CAM','BAH',1,1),
    (19,'CFC','PAL',1,3),
    (19,'CHA','FLA',0,4),
    (19,'INT','CRU',1,2),
    (19,'SAO','CAP',1,2),
    (19,'COR','REM',3,0),
]

JOGOS_R20 = [
    (20,'CAP','INT',2,0),
    (20,'SAN','CHA',2,2),
    (20,'VAS','MIR',1,1),
    (20,'BAH','COR',1,1),
    (20,'CRU','BOT',0,1),
    (20,'FLA','SAO',1,1),
    (20,'GRE','FLU',1,1),
    (20,'RBB','CFC',0,0),
    (20,'PAL','CAM',1,2),
    (20,'REM','VIT',2,0),
]

JOGOS_R21 = [
    (21,'INT','FLA',1,1),
    (21,'MIR','REM',2,1),
    (21,'FLU','BAH',0,0),
    (21,'VIT','PAL',0,4),
    (21,'CAM','RBB',1,0),
    (21,'BOT','GRE',3,2),
    #(21,'CHA','VAS',?,?),
    (21,'SAO','SAN',1,2),
    (21,'COR','CAP',0,0),
    (21,'CFC','CRU',0,1),
]

JOGOS_R22 = [
    (22,'GRE','SAO',2,1),
    (22,'REM','CAM',2,2),
    (22,'CFC','CHA',2,1),
    (22,'BOT','FLU',1,1),
    (22,'CRU','MIR',3,1),
    (22,'PAL','INT',0,0),
    (22,'BAH','VAS',0,0),
    (22,'SAN','CAP',0,2),
    (22,'RBB','COR',0,2),
    (22,'FLA','VIT',2,0),
]

JOGOS_R23 = [
    (23,'FLU','PAL',3,2),
    (23,'CAP','RBB',1,1),
    (23,'SAO','CFC',1,1),
    (23,'CHA','BAH',3,3),
    (23,'CAM','GRE',3,0),
    (23,'VAS','SAN',0,3),
    (23,'VIT','BOT',1,0),
    (23,'MIR','FLA',1,5),
    (23,'COR','CRU',1,2),
    (23,'INT','REM',1,1),
]

JOGOS_R24 = [
    (24,'FLU','REM',2,1),
    (24,'INT','CAM',0,0),
    (24,'CRU','FLA',2,1),
    (24,'PAL','VAS',4,1),
    (24,'RBB','GRE',1,0),
    (24,'VIT','BAH',0,2),
    (24,'CHA','SAO',1,0),
    (24,'SAN','MIR',1,1),
    (24,'CFC','COR',2,1),
    (24,'BOT','CAP',2,3),
]

JOGOS_R25 = [
    (25,'CAM','VIT',2,1),
    (25,'SAO','RBB',2,1),
    (25,'VAS','CRU',3,1),
    (25,'CAP','FLU',3,3),
    (25,'COR','SAN',0,1),
    (25,'FLA','BOT',3,0),
    (25,'GRE','CHA',3,1),
    (25,'MIR','PAL',1,1),
    (25,'BAH','INT',3,2),
    (25,'REM','CFC',2,3),
]

JOGOS_R26 = [
    (26,'RBB','BAH',2,3),
    (26,'SAO','CAM',2,0),
    (26,'FLU','VAS',1,0),
    (26,'CFC','MIR',1,2),
    (26,'CRU','CAP',3,1),
    (26,'INT','SAN',2,3),
    (26,'REM','FLA',0,1),
    (26,'BOT','PAL',0,0),
    (26,'COR','CHA',1,2),
    (26,'VIT','GRE',1,0),
]

JOGOS_R27 = [
    (27,'CFC','CAP',3,3),
    (27,'CAM','FLU',3,1),
    (27,'GRE','VAS',1,2),
    (27,'CHA','INT',1,2),
    (27,'PAL','SAO',2,0),
    (27,'BOT','RBB',1,1),
    (27,'SAN','CRU',2,1),
    (27,'MIR','VIT',2,2),
    (27,'FLA','COR',2,1),
    (27,'BAH','REM',2,1),
]

JOGOS_R28 = [
    (28,'CAM','CHA',1,1),
    (28,'MIR','BOT',2,0),
    (28,'REM','SAN',1,2),
    (28,'VAS','CFC',5,0),
    (28,'SAO','INT',1,0),
    (28,'GRE','PAL',0,0),
    (28,'COR','FLU',1,3),
    (28,'VIT','CRU',1,3),
    (28,'FLA','RBB',2,1),
    (28,'CAP','BAH',2,1),
]

JOGOS_R29 = [
    (29,'INT','COR',2,1),
    (29,'RBB','MIR',1,1),
    (29,'REM','GRE',1,1),
    (29,'VIT','CHA',4,0),
    (29,'BOT','VAS',1,2),
    (29,'CRU','SAO',2,0),
    (29,'SAN','FLA',2,2),
    (29,'CAP','CAM',2,2),
    (29,'FLU','CFC',4,0),
    (29,'PAL','BAH',1,0),
]

JOGOS_R30 = [
    (30,'VAS','REM',0,0),
    (30,'SAO','VIT',2,1),
    (30,'CAM','SAN',2,1),
    (30,'FLA','FLU',1,0),
    (30,'GRE','INT',1,1),
    (30,'PAL','COR',4,1),
    (30,'BAH','MIR',1,0),
    (30,'CFC','BOT',1,1),
    (30,'CHA','CAP',1,4),
    (30,'RBB','CRU',4,2),
]

JOGOS_R31 = [
    (31,'MIR','INT',2,0),
    (31,'BOT','CHA',1,0),
    (31,'CAM','CFC',1,0),
    (31,'CAP','PAL',0,2),
    (31,'SAO','VAS',1,0),
    (31,'FLU','SAN',1,2),
    (31,'GRE','CRU',2,4),
    (31,'BAH','FLA',0,3),
    (31,'REM','RBB',1,1),
    (31,'COR','VIT',1,1)
]

JOGOS_R32 = [
    (32,'CRU','REM',1,0),
    (32,'INT','BOT',3,0),
    (32,'MIR','SAO',1,1),
    (32,'VIT','CAP',1,1),
    (32,'PAL','RBB',2,0),
    (32,'VAS','COR',3,2),
    (32,'CHA','FLU',2,1),
    (32,'SAN','BAH',1,1),
    (32,'CFC','GRE',0,1),
    (32,'FLA','CAM',3,2)
]

JOGOS_R33 = [
    (33,'FLU','INT',2,1),
    (33,'RBB','CHA',3,0),
    (33,'SAN','PAL',0,0),
    (33,'BAH','SAO',3,2),
    (33,'REM','BOT',2,1),
    (33,'CAM','CRU',0,1),
    (33,'VAS','FLA',1,4),
    (33,'CFC','VIT',0,0),
    (33,'GRE','CAP',2,0),
    (33,'COR','MIR',2,0)
]

JOGOS_R34 = [
    (34,'RBB','SAN',2,1),
    (34,'CHA','MIR',1,2),
    (34,'INT','CFC',1,0),
    (34,'CAP','VAS',1,3),
    (34,'FLA','GRE',3,1),
    (34,'BOT','CAM',1,0),
    (34,'SAO','COR',0,0),
    (34,'PAL','REM',6,1),
    (34,'CRU','BAH',3,2),
    (34,'VIT','FLU',0,3)
]

JOGOS_R35 = [
    (35,'MIR','CAM',0,0),
    (35,'CFC','SAN',0,0),
    (35,'REM','CHA',0,0),
    (35,'CRU','PAL',0,0),
    (35,'FLA','CAP',0,0),
    (35,'COR','BOT',0,0),
    (35,'GRE','BAH',0,0),
    (35,'VAS','INT',0,0),
    (35,'VIT','RBB',0,0),
    (35,'SAO','FLU',0,0)
]

JOGOS_R36 = [
    (36,'PAL','FLA',0,0),
    (36,'BAH','CFC',0,0),
    (36,'CHA','CRU',0,0),
    (36,'CAM','COR',0,0),
    (36,'CAP','REM',0,0),
    (36,'FLU','MIR',0,0),
    (36,'RBB','VAS',0,0),
    (36,'INT','VIT',0,0),
    (36,'BOT','SAO',0,0),
    (36,'SAN','GRE',0,0)
]

JOGOS_R37 = [
    (37,'CAM','VAS',0,0),
    (37,'BOT','BAH',0,0),
    (37,'CHA','PAL',0,0),
    (37,'COR','GRE',0,0),
    (37,'CFC','FLA',0,0),
    (37,'FLU','CRU',0,0),
    (37,'INT','RBB',0,0),
    (37,'MIR','CAP',0,0),
    (37,'SAO','REM',0,0),
    (37,'VIT','SAN',0,0)
]

JOGOS_R38 = [
    (38,'CAP','SAO',0,0),
    (38,'BAH','CAM',0,0),
    (38,'CRU','INT',0,0),
    (38,'FLA','CHA',0,0),
    (38,'GRE','MIR',0,0),
    (38,'PAL','CFC',0,0),
    (38,'RBB','FLU',0,0),
    (38,'REM','COR',0,0),
    (38,'SAN','BOT',0,0),
    (38,'VAS','VIT',0,0)
]

# ── Monta o DataFrame de todos os jogos ──────────────────────────────
COLUNAS = ['rodada','mandante','visitante','gols_mandante','gols_visitante']

TODAS_RODADAS = {
    1: JOGOS_R1, 2: JOGOS_R2, 3: JOGOS_R3, 4: JOGOS_R4,
    5: JOGOS_R5, 6: JOGOS_R6, 7: JOGOS_R7, 8: JOGOS_R8,
    9: JOGOS_R9, 10: JOGOS_R10, 11: JOGOS_R11, 12: JOGOS_R12,
    13: JOGOS_R13, 14: JOGOS_R14, 15: JOGOS_R15, 16: JOGOS_R16,
    17: JOGOS_R17, 18: JOGOS_R18, 19: JOGOS_R19, 20: JOGOS_R20,
    21: JOGOS_R21, 22: JOGOS_R22, 23: JOGOS_R23, 24: JOGOS_R24,
    25: JOGOS_R25, 26: JOGOS_R26, 27: JOGOS_R27, 28: JOGOS_R28,
    29: JOGOS_R29, 30: JOGOS_R30, 31: JOGOS_R31, 32: JOGOS_R32,
    33: JOGOS_R33, 34: JOGOS_R34, 35: JOGOS_R35, 36: JOGOS_R36,
    37: JOGOS_R37, 38: JOGOS_R38
}

todos = []
for jogos in TODAS_RODADAS.values():
    todos.extend([j for j in jogos if '?' not in str(j)])

df_jogos_full = pd.DataFrame(todos, columns=COLUNAS)
df_jogos_full['mandante']       = df_jogos_full['mandante'].apply(normalizar_time)
df_jogos_full['visitante']      = df_jogos_full['visitante'].apply(normalizar_time)
df_jogos_full['gols_mandante']  = df_jogos_full['gols_mandante'].astype(int)
df_jogos_full['gols_visitante'] = df_jogos_full['gols_visitante'].astype(int)

# ── Aplica filtro ATE_RODADA (definido na Célula 3) ──────────────────
if ATE_RODADA is not None:
    df_jogos = df_jogos_full[df_jogos_full['rodada'] <= ATE_RODADA].copy()
    RODADA_ATUAL = ATE_RODADA
else:
    df_jogos = df_jogos_full.copy()
    RODADA_ATUAL = int(df_jogos['rodada'].max())

print(f"{len(df_jogos)} jogos carregados | Rodadas 1 a {RODADA_ATUAL}")

# ── Recalcula classificação oficial ──────────────────────────────────
tabela = defaultdict(lambda: {"p":0,"j":0,"v":0,"e":0,"d":0,"gp":0,"gc":0})
for _, row in df_jogos.iterrows():
    m, v   = row['mandante'], row['visitante']
    gm, gv = int(row['gols_mandante']), int(row['gols_visitante'])
    tabela[m]['j'] += 1; tabela[v]['j'] += 1
    tabela[m]['gp'] += gm; tabela[m]['gc'] += gv
    tabela[v]['gp'] += gv; tabela[v]['gc'] += gm
    if gm > gv:
        tabela[m]['v'] += 1; tabela[m]['p'] += 3; tabela[v]['d'] += 1
    elif gv > gm:
        tabela[v]['v'] += 1; tabela[v]['p'] += 3; tabela[m]['d'] += 1
    else:
        tabela[m]['e'] += 1; tabela[m]['p'] += 1
        tabela[v]['e'] += 1; tabela[v]['p'] += 1

times_ord = sorted(
    tabela.keys(),
    key=lambda t: (tabela[t]['p'], tabela[t]['v'], tabela[t]['gp']-tabela[t]['gc']),
    reverse=True
)
payload_oficial = []
for pos, time in enumerate(times_ord, 1):
    s = tabela[time]
    payload_oficial.append({
        "serie":"A", "rodada":RODADA_ATUAL, "pos":pos, "time":time,
        "p":s['p'], "j":s['j'], "v":s['v'], "e":s['e'], "d":s['d'],
        "gp":s['gp'], "gc":s['gc'], "sg":s['gp']-s['gc'],
    })

r = requests.post(
    f"{SUPABASE_URL}/rest/v1/oficial?on_conflict=serie,rodada,time",
    headers=headers_sb,
    json=payload_oficial
)
if r.status_code in (200, 201):
    print(f"Classificação oficial da Rodada {RODADA_ATUAL} enviada ao Supabase!")
else:
    print(f"Erro: {r.status_code} — {r.text}")

# ── Cálculo de forças híbridas (FM26 + Tabela Real) ──────────────────

# Força FM26: média dos 16 melhores jogadores por time, ao cubo
forcas_fm = {}
for time in TIMES_SERIE_A:
    df_time = df_fm[df_fm['Time'] == time].copy()
    if len(df_time) == 0:
        forcas_fm[time] = 1.0
        continue
    col_ca = [c for c in df_fm.columns if 'CA' in str(c) or 'ca' in str(c) or 'Rating' in str(c)][0]
    top16 = df_time[col_ca].nlargest(16).mean()
    forcas_fm[time] = top16 ** 3

media_fm = np.mean(list(forcas_fm.values()))
forcas_fm = {t: v / media_fm for t, v in forcas_fm.items()}

# Pontos reais e força real por desempenho na tabela
pontos_reais = {}
for time in TIMES_SERIE_A:
    jogos_time = df_jogos[(df_jogos['mandante'] == time) | (df_jogos['visitante'] == time)]
    p, j, v, e, d, gp, gc = 0, 0, 0, 0, 0, 0, 0
    for _, row in jogos_time.iterrows():
        gm = row['gols_mandante']
        gv = row['gols_visitante']
        if row['mandante'] == time:
            gp += gm; gc += gv
        else:
            gp += gv; gc += gm
        j += 1
        if (row['mandante'] == time and gm > gv) or (row['visitante'] == time and gv > gm):
            p += 3; v += 1
        elif gm == gv:
            p += 1; e += 1
        else:
            d += 1
    pontos_reais[time] = {'P': p, 'J': j, 'V': v, 'E': e, 'D': d, 'GP': gp, 'GC': gc}

# Força real: aproveitamento (60%) + SG normalizado (40%)
forcas_reais = {}
for time in TIMES_SERIE_A:
    s = pontos_reais[time]
    aprov = s['P'] / (s['J'] * 3) if s['J'] > 0 else 0.0
    sg = s['GP'] - s['GC']
    forcas_reais[time] = aprov * 0.6 + sg * 0.4

min_r = min(forcas_reais.values())
max_r = max(forcas_reais.values())
if max_r > min_r:
    forcas_reais = {t: (v - min_r) / (max_r - min_r) + 0.5 for t, v in forcas_reais.items()}
else:
    forcas_reais = {t: 1.0 for t in TIMES_SERIE_A}

media_r = np.mean(list(forcas_reais.values()))
forcas_reais = {t: v / media_r for t, v in forcas_reais.items()}

# Força final híbrida: 50% FM26 + 50% Real
forcas_finais = {t: (forcas_fm[t] + forcas_reais[t]) / 2 for t in TIMES_SERIE_A}

print("=" * 65)
print(f"{'FORÇAS E SITUAÇÃO ATUAL':^65}")
print("=" * 65)
print(f"{'Time':<30} {'Força':>7} {'Pts':>5} {'J':>4} {'SG':>5}")
print("-" * 65)
for time in sorted(TIMES_SERIE_A, key=lambda t: forcas_finais[t], reverse=True):
    s = pontos_reais[time]
    print(f"  {time:<28} {forcas_finais[time]:>7.4f} {s['P']:>5} {s['J']:>4} {s['GP']-s['GC']:>+5}")

In [ ]:
# ============================================================
# CÉLULA 5 — MOTOR MONTE CARLO (10.000 SIMULAÇÕES)
# ============================================================

# ── Monta a lista de jogos restantes (todos os 380 menos os já jogados)
def gerar_todos_jogos():
    """Gera os 380 jogos do turno único (todos jogam contra todos)."""
    jogos = []
    for i, m in enumerate(TIMES_SERIE_A):
        for j, v in enumerate(TIMES_SERIE_A):
            if i != j:
                jogos.append((m, v))
    return jogos

todos_jogos = gerar_todos_jogos()  # 380 jogos

# Jogos já realizados (par mandante-visitante)
jogados = set(zip(df_jogos['mandante'], df_jogos['visitante']))

jogos_restantes = [(m, v) for m, v in todos_jogos if (m, v) not in jogados]

print(f"Total de jogos no campeonato : {len(todos_jogos)}")
print(f"Jogos já realizados          : {len(jogados)}")
print(f"Jogos restantes a simular    : {len(jogos_restantes)}")

# ── Motor de simulação
print(f"\nIniciando {NUM_SIMULACOES:,} simulações Monte Carlo...")

# Matriz de contagem de posições: times x posições (1 a 20)
contagem_posicoes = {t: np.zeros(20, dtype=int) for t in TIMES_SERIE_A}
soma_pontos       = {t: 0.0 for t in TIMES_SERIE_A}

for sim in range(NUM_SIMULACOES):

    # Fator de Fase: UMA única perturbação por simulação inteira
    # Isso é o que cria o espalhamento realista na tabela final
    fase = np.random.normal(FATOR_FASE_MEDIA, FATOR_FASE_DESVIO)
    fase = max(0.5, min(fase, 1.5))  # limita para não gerar absurdos

    # Copia pontos reais
    pts_sim = {t: pontos_reais[t]['P'] for t in TIMES_SERIE_A}

    # Simula jogos restantes
    for mandante, visitante in jogos_restantes:
        fm = forcas_finais[mandante]
        fv = forcas_finais[visitante]

        # Lambdas de Poisson com vantagem do mandante e fator de fase
        lambda_m = max(LAMBDA_MIN, min(MU * fm * VANTAGEM_MANDANTE * fase, LAMBDA_MAX))
        lambda_v = max(LAMBDA_MIN, min(MU * fv * fase, LAMBDA_MAX))

        gols_m = np.random.poisson(lambda_m)
        gols_v = np.random.poisson(lambda_v)

        if gols_m > gols_v:
            pts_sim[mandante] += 3
        elif gols_v > gols_m:
            pts_sim[visitante] += 3
        else:
            pts_sim[mandante] += 1
            pts_sim[visitante] += 1

    # Classifica os times nesta simulação
    # Critério de desempate: pontos → force (proxy de vitórias)
    ordem = sorted(
        TIMES_SERIE_A,
        key=lambda t: (pts_sim[t], forcas_finais[t]),
        reverse=True
    )

    for pos, time in enumerate(ordem):
        contagem_posicoes[time][pos] += 1
        soma_pontos[time] += pts_sim[time]

    if (sim + 1) % 2000 == 0:
        print(f"   {sim+1:>6,} / {NUM_SIMULACOES:,} simulações concluídas...")

print(f"\n{NUM_SIMULACOES:,} simulações concluídas!")

In [ ]:
# ============================================================
# CÉLULA 6 — TABELA FINAL DE PROBABILIDADES
# ============================================================

resultados = []

for time in TIMES_SERIE_A:
    probs = contagem_posicoes[time] / NUM_SIMULACOES
    pts_medio = soma_pontos[time] / NUM_SIMULACOES

    # Posição mais provável (onde o time mais apareceu)
    pos_media = int(np.argmax(contagem_posicoes[time])) + 1

    # Probabilidades por zona
    p_campeao      = probs[0]
    p_libertadores = probs[0:4].sum()   # top 4 = fase de grupos
    p_pre_liberta  = probs[4]           # 5º = pré-Libertadores
    p_sulamerica   = probs[5:12].sum()  # 6º ao 12º = Sul-Americana
    p_rebaixamento = probs[16:20].sum() # 17º ao 20º

    # Status por posição média (regra rígida do projeto)
    if pos_media == 1:
        status = '🏆 Campeão'
    elif pos_media <= 4:
        status = '🟢 Libertadores'
    elif pos_media == 5:
        status = '🟡 Pré-Libertadores'
    elif pos_media <= 12:
        status = '🔵 Sul-Americana'
    elif pos_media <= 16:
        status = '⚪ Meio de Tabela'
    else:
        status = '🔴 Rebaixamento'

    s = pontos_reais[time]
    resultados.append({
        'Time'             : time,
        'Pts Reais'        : s['P'],
        'J Reais'          : s['J'],
        'Pts Médios (proj)': round(pts_medio, 1),
        'Pos. Mais Provável': pos_media,
        'Status'           : status,
        '🏆 Campeão'       : p_campeao,
        '🟢 Libertadores'  : p_libertadores,
        '🟡 Pré-Lib.'      : p_pre_liberta,
        '🔵 Sul-Americana' : p_sulamerica,
        '🔴 Rebaixamento'  : p_rebaixamento,
        # Probabilidades por posição (1 a 20)
        **{f'P{i+1:02d}': probs[i] for i in range(20)}
    })

df_resultado = pd.DataFrame(resultados)
df_resultado = df_resultado.sort_values('Pos. Mais Provável').reset_index(drop=True)
df_resultado.index += 1
df_resultado.insert(0, 'Pos', df_resultado.index)

print("=" * 65)
print(f"{'PROJEÇÃO FINAL — BRASILEIRÃO SÉRIE A 2026':^65}")
print("=" * 65)
print(f"{'Pos':<5}{'Time':<26}{'Pts R':>6}{'Pts Proj':>9}{'Status'}")
print("-" * 65)
for _, row in df_resultado.iterrows():
    print(f"  {int(row['Pos']):<4}{row['Time']:<26}{int(row['Pts Reais']):>5}"
          f"  {row['Pts Médios (proj)']:>7.1f}  {row['Status']}")

In [ ]:
# ============================================================
# CÉLULA 7 — EXPORTAÇÃO EXCEL PREMIUM COM XLSXWRITER
# ============================================================

RODADA_ATUAL = int(df_jogos['rodada'].max())
ARQUIVO_SAIDA = f'SimulacaoSerieA_2026_Rodada{RODADA_ATUAL}.xlsx'

writer   = pd.ExcelWriter(ARQUIVO_SAIDA, engine='xlsxwriter')
workbook = writer.book

# ════════════════════════════════════════════════════════
# FORMATOS GLOBAIS
# ════════════════════════════════════════════════════════
fmt_titulo = workbook.add_format({
    'bold': True, 'font_size': 14, 'font_color': '#FFFFFF',
    'bg_color': '#112233', 'align': 'center', 'valign': 'vcenter',
    'border': 0
})
fmt_subtitulo = workbook.add_format({
    'italic': True, 'font_size': 10, 'font_color': '#888888',
    'align': 'center', 'valign': 'vcenter'
})
fmt_header = workbook.add_format({
    'bold': True, 'font_color': '#FFFFFF', 'bg_color': '#112233',
    'align': 'center', 'valign': 'vcenter', 'border': 1,
    'text_wrap': True
})
fmt_pos = workbook.add_format({
    'bold': True, 'align': 'center', 'valign': 'vcenter',
    'border': 1, 'font_size': 12
})
fmt_num = workbook.add_format({
    'align': 'center', 'valign': 'vcenter', 'border': 1
})
fmt_pct = workbook.add_format({
    'num_format': '0.0%', 'align': 'center',
    'valign': 'vcenter', 'border': 1
})
fmt_pts_proj = workbook.add_format({
    'bold': True, 'num_format': '0.0', 'align': 'center',
    'valign': 'vcenter', 'border': 1, 'font_size': 12
})

# Formatos por zona (Status)
def fmt_zona(bg, fg='#000000', bold=False):
    return workbook.add_format({
        'bg_color': bg, 'font_color': fg, 'bold': bold,
        'align': 'left', 'valign': 'vcenter', 'border': 1,
        'indent': 1, 'font_size': 11
    })

fmt_campeao   = fmt_zona('#FFD700', '#7B4F00', bold=True)
fmt_liberta   = fmt_zona('#D5F5E3', '#1E8449', bold=True)
fmt_prelib    = fmt_zona('#D6EAF8', '#1A5276')
fmt_sula      = fmt_zona('#EBF5FB', '#2E86C1')
fmt_meio      = fmt_zona('#F8F9FA', '#555555')
fmt_rebaixado = fmt_zona('#FADBD8', '#C0392B', bold=True)

fmt_status_map = {
    '🏆 Campeão'        : workbook.add_format({'bg_color':'#FFD700','font_color':'#7B4F00','bold':True,'align':'center','valign':'vcenter','border':1}),
    '🟢 Libertadores'   : workbook.add_format({'bg_color':'#D5F5E3','font_color':'#1E8449','bold':True,'align':'center','valign':'vcenter','border':1}),
    '🟡 Pré-Libertadores': workbook.add_format({'bg_color':'#D6EAF8','font_color':'#1A5276','align':'center','valign':'vcenter','border':1}),
    '🔵 Sul-Americana'  : workbook.add_format({'bg_color':'#EBF5FB','font_color':'#2E86C1','align':'center','valign':'vcenter','border':1}),
    '⚪ Meio de Tabela'  : workbook.add_format({'bg_color':'#F8F9FA','font_color':'#555555','align':'center','valign':'vcenter','border':1}),
    '🔴 Rebaixamento'   : workbook.add_format({'bg_color':'#FADBD8','font_color':'#C0392B','bold':True,'align':'center','valign':'vcenter','border':1}),
}

fmt_time_map = {
    '🏆 Campeão'        : fmt_campeao,
    '🟢 Libertadores'   : fmt_liberta,
    '🟡 Pré-Libertadores': fmt_prelib,
    '🔵 Sul-Americana'  : fmt_sula,
    '⚪ Meio de Tabela'  : fmt_meio,
    '🔴 Rebaixamento'   : fmt_rebaixado,
}

fmt_num_zona = {}
for status, bg, fg in [
    ('🏆 Campeão',         '#FFD700','#7B4F00'),
    ('🟢 Libertadores',    '#D5F5E3','#1E8449'),
    ('🟡 Pré-Libertadores','#D6EAF8','#1A5276'),
    ('🔵 Sul-Americana',   '#EBF5FB','#2E86C1'),
    ('⚪ Meio de Tabela',   '#F8F9FA','#555555'),
    ('🔴 Rebaixamento',    '#FADBD8','#C0392B'),
]:
    fmt_num_zona[status] = workbook.add_format({
        'bg_color': bg, 'font_color': fg,
        'align': 'center', 'valign': 'vcenter', 'border': 1
    })

# ════════════════════════════════════════════════════════
# ABA 1 — PROJEÇÃO FINAL
# ════════════════════════════════════════════════════════
colunas_principais = [
    'Pos', 'Time', 'Pts Reais', 'J Reais',
    'Pts Médios (proj)', 'Pos. Mais Provável', 'Status',
    '🏆 Campeão', '🟢 Libertadores', '🟡 Pré-Lib.',
    '🔵 Sul-Americana', '🔴 Rebaixamento'
]
df_aba1 = df_resultado[colunas_principais]
df_aba1.to_excel(writer, sheet_name='Projeção Final', index=False, startrow=3)
ws1 = writer.sheets['Projeção Final']

ws1.merge_range('A1:L1',
    f'🏆 BRASILEIRÃO SÉRIE A 2026 — PROJEÇÃO MONTE CARLO ({NUM_SIMULACOES:,} simulações)',
    fmt_titulo)
ws1.merge_range('A2:L2',
    f'Base: {RODADA_ATUAL} rodadas disputadas  |  Fator de Fase: normal(1.0, 0.20)  |  Vantagem mandante: 1.15x',
    fmt_subtitulo)
ws1.set_row(0, 34)
ws1.set_row(1, 18)
ws1.set_row(2, 6)

headers = ['Pos', 'Time', 'Pts\nReais', 'J\nReais',
           'Pts Proj.', 'Pos.\nMédia', 'Status',
           'Campeão', 'Liberta-\ndores', 'Pré-Lib.',
           'Sul-\nAmericana', 'Reba-\nxamento']
for c, h in enumerate(headers):
    ws1.write(3, c, h, fmt_header)
ws1.set_row(3, 30)

for i, row in df_resultado.iterrows():
    r = i + 3  # linha no excel (0-indexed, +3 por título+sub+espaço+header)
    st = row['Status']
    fmt_t = fmt_time_map.get(st, fmt_meio)
    fmt_n = fmt_num_zona.get(st, fmt_num)
    fmt_s = fmt_status_map.get(st, fmt_num)

    ws1.write(r, 0,  int(row['Pos']),                fmt_pos)
    ws1.write(r, 1,  row['Time'],                     fmt_t)
    ws1.write(r, 2,  int(row['Pts Reais']),           fmt_n)
    ws1.write(r, 3,  int(row['J Reais']),             fmt_n)
    ws1.write(r, 4,  row['Pts Médios (proj)'],        fmt_pts_proj)
    ws1.write(r, 5,  int(row['Pos. Mais Provável']),  fmt_n)
    ws1.write(r, 6,  st,                              fmt_s)
    ws1.write(r, 7,  row['🏆 Campeão'],               fmt_pct)
    ws1.write(r, 8,  row['🟢 Libertadores'],          fmt_pct)
    ws1.write(r, 9,  row['🟡 Pré-Lib.'],              fmt_pct)
    ws1.write(r, 10, row['🔵 Sul-Americana'],         fmt_pct)
    ws1.write(r, 11, row['🔴 Rebaixamento'],          fmt_pct)
    ws1.set_row(r, 22)

ws1.set_column('A:A', 5)
ws1.set_column('B:B', 26)
ws1.set_column('C:D', 8)
ws1.set_column('E:F', 10)
ws1.set_column('G:G', 22)
ws1.set_column('H:L', 12)
ws1.freeze_panes(4, 2)

# ════════════════════════════════════════════════════════
# ABA 2 — MAPA DE CALOR (probabilidades por posição)
# ════════════════════════════════════════════════════════
colunas_pos = ['Pos', 'Time', 'Status'] + [f'P{i+1:02d}' for i in range(20)]
df_aba2 = df_resultado[colunas_pos]
df_aba2.to_excel(writer, sheet_name='Mapa de Calor', index=False, startrow=3)
ws2 = writer.sheets['Mapa de Calor']

ws2.merge_range(0, 0, 0, 22,
    '🗺️ MAPA DE CALOR — PROBABILIDADE POR POSIÇÃO FINAL',
    fmt_titulo)
ws2.merge_range(1, 0, 1, 22,
    'Cada célula = probabilidade do time terminar naquela posição  |  Escala: Branco (0%) → Amarelo → Verde (100%)',
    fmt_subtitulo)
ws2.set_row(0, 34); ws2.set_row(1, 18); ws2.set_row(2, 6)

headers2 = ['Pos', 'Time', 'Status'] + [str(i+1) for i in range(20)]
for c, h in enumerate(headers2):
    ws2.write(3, c, h, fmt_header)
ws2.set_row(3, 22)

for i, row in df_resultado.iterrows():
    r = i + 3
    st = row['Status']
    ws2.write(r, 0, int(row['Pos']),  fmt_pos)
    ws2.write(r, 1, row['Time'],      fmt_time_map.get(st, fmt_meio))
    ws2.write(r, 2, st,               fmt_status_map.get(st, fmt_num))
    for p in range(20):
        ws2.write(r, p + 3, row[f'P{p+1:02d}'], fmt_pct)
    ws2.set_row(r, 20)

# Mapa de calor 3-color scale em toda a matriz de probabilidades
# Branco (0%) → Amarelo (50%) → Verde escuro (100%)
ws2.conditional_format(4, 3, 23, 22, {
    'type'     : '3_color_scale',
    'min_color': '#FFFFFF',
    'mid_color': '#FFEB84',
    'max_color': '#63BE7B',
    'min_type' : 'num', 'min_value': 0,
    'mid_type' : 'percentile', 'mid_value': 50,
    'max_type' : 'num', 'max_value': 1,
})

ws2.set_column('A:A', 5)
ws2.set_column('B:B', 26)
ws2.set_column('C:C', 22)
ws2.set_column('D:W', 6)
ws2.freeze_panes(4, 3)

# ════════════════════════════════════════════════════════
# ABA 3 — TABELA REAL ATUAL
# ════════════════════════════════════════════════════════
rows_tabela = []
for time, s in sorted(pontos_reais.items(),
                       key=lambda x: (x[1]['P'], x[1]['V'], x[1]['GP']-x[1]['GC']),
                       reverse=True):
    rows_tabela.append({
        'Time': time, 'P': s['P'], 'J': s['J'],
        'V': s['V'], 'E': s['E'], 'D': s['D'],
        'GP': s['GP'], 'GC': s['GC'], 'SG': s['GP']-s['GC'],
    })
df_tabela = pd.DataFrame(rows_tabela)
df_tabela.insert(0, 'Pos', range(1, 21))

df_tabela.to_excel(writer, sheet_name='Tabela Atual', index=False, startrow=3)
ws3 = writer.sheets['Tabela Atual']

ws3.merge_range('A1:J1',
    f'📊 CLASSIFICAÇÃO REAL — APÓS RODADA {RODADA_ATUAL}',
    fmt_titulo)
ws3.set_row(0, 34); ws3.set_row(1, 6)

for c, h in enumerate(['Pos','Time','P','J','V','E','D','GP','GC','SG']):
    ws3.write(3, c, h, fmt_header)
ws3.set_row(3, 22)

for i, row in df_tabela.iterrows():
    r = i + 4
    pos = int(row['Pos'])
    if pos == 1:
        bg, fg = '#FFD700', '#7B4F00'
    elif pos <= 4:
        bg, fg = '#D5F5E3', '#1E8449'
    elif pos == 5:
        bg, fg = '#D6EAF8', '#1A5276'
    elif pos <= 12:
        bg, fg = '#EBF5FB', '#2E86C1'
    elif pos >= 17:
        bg, fg = '#FADBD8', '#C0392B'
    else:
        bg, fg = '#F8F9FA', '#555555'

    fmt_l = workbook.add_format({'bg_color':bg,'font_color':fg,'bold':pos<=4 or pos>=17,'align':'center','valign':'vcenter','border':1})
    fmt_nome = workbook.add_format({'bg_color':bg,'font_color':fg,'bold':pos<=4 or pos>=17,'align':'left','valign':'vcenter','border':1,'indent':1})

    ws3.write(r, 0, pos,           fmt_l)
    ws3.write(r, 1, row['Time'],   fmt_nome)
    for c_idx, campo in enumerate(['P','J','V','E','D','GP','GC','SG'], start=2):
        ws3.write(r, c_idx, int(row[campo]), fmt_l)
    ws3.set_row(r, 22)

ws3.set_column('A:A', 5)
ws3.set_column('B:B', 26)
ws3.set_column('C:J', 7)
ws3.freeze_panes(4, 2)

# ── Salva e baixa
writer.close()
files.download(ARQUIVO_SAIDA)
print(f"\n✅ '{ARQUIVO_SAIDA}' gerado e baixado!")
print(f"   → 3 abas: Projeção Final | Mapa de Calor | Tabela Atual")
print(f"   → {NUM_SIMULACOES:,} simulações | {len(jogos_restantes)} jogos simulados por rodada")

In [ ]:
from google.colab import userdata
# =====================================================================
# CÉLULA 8 — UPLOAD AUTOMÁTICO DA PROJEÇÃO PARA O SUPABASE
# =====================================================================
# Preencha sua SUPABASE_SECRET_KEY abaixo antes de rodar.
# A SUPABASE_URL já está definida.
# =====================================================================

import requests

SUPABASE_URL        = userdata.get("SUPABASE_URL")
SUPABASE_SECRET_KEY = userdata.get("SUPABASE_SECRET_KEY")

headers = {
    "apikey"       : SUPABASE_SECRET_KEY,
    "Authorization": f"Bearer {SUPABASE_SECRET_KEY}",
    "Content-Type" : "application/json",
    "Prefer"       : "resolution=merge-duplicates",
}

# ── Monta o payload ──────────────────────────────────────────────────
payload = []
for _, row in df_resultado.iterrows():
    payload.append({
        "serie"            : "A",
        "rodada"           : int(RODADA_ATUAL),
        "pos"              : int(row["Pos"]),
        "time"             : row["Time"],
        "pts_reais"        : int(row["Pts Reais"]),
        "j_reais"          : int(row["J Reais"]),
        "pts_proj"         : float(row["Pts Médios (proj)"]),
        "pos_mais_provavel": int(row["Pos. Mais Provável"]),
        "status"           : row["Status"],
        "p_campeao"        : float(row["🏆 Campeão"]),
        "p_acesso"         : float(row["🟢 Libertadores"]),
        "p_rebaixamento"   : float(row["🔴 Rebaixamento"]),
    })

# ── Envia para o Supabase ────────────────────────────────────────────
url = f"{SUPABASE_URL}/rest/v1/projecao?on_conflict=serie,rodada,time"
response = requests.post(url, headers=headers, json=payload)

if response.status_code in (200, 201):
    print(f"{len(payload)} times enviados para o Supabase!")
    print(f"   Série A — Rodada {RODADA_ATUAL}")
else:
    print(f"Erro {response.status_code}: {response.text}")

In [ ]:
# =====================================================================
# CÉLULA 9 — VALIDAÇÃO ESTATÍSTICA COM CORREÇÃO DE DIXON-COLES
# =====================================================================
# Dixon & Coles (1997) introduzem um parâmetro rho que corrige a
# subestimação de empates e resultados de placar baixo no modelo
# Poisson independente padrão.
#
# A correção τ(g_m, g_v, λ_m, λ_v, ρ) é aplicada nos placares:
#   (0,0), (1,0), (0,1) e (1,1)
# Para todos os outros placares, τ = 1.0 (sem correção).
#
# Referência: Dixon & Coles (1997) — "Modelling Association Football
# Scores and Inefficiencies in the Football Betting Market"
# =====================================================================

import numpy as np
from scipy.stats import poisson

MU       = 1.3    # média histórica de gols do Brasileirão por jogo
CLIP_MIN = 0.3
CLIP_MAX = 4.0
N_GOLS   = 10
RHO      = -0.13  # parâmetro de correlação Dixon-Coles
                  # valor típico para futebol: entre -0.10 e -0.15
                  # negativo pois gols mandante e visitante são
                  # negativamente correlacionados em placares baixos

def tau(g_m, g_v, lm, lv, rho):
    """
    Fator de correção de Dixon-Coles para placares baixos.
    Ajusta a probabilidade conjunta P(G_m = g_m, G_v = g_v).
    """
    if g_m == 0 and g_v == 0:
        return 1 - lm * lv * rho
    elif g_m == 1 and g_v == 0:
        return 1 + lv * rho
    elif g_m == 0 and g_v == 1:
        return 1 + lm * rho
    elif g_m == 1 and g_v == 1:
        return 1 - rho
    else:
        return 1.0

def calcular_forcas(df_jogos_ate_rodada, df_fm, times):
    """Recalcula forças híbridas usando apenas jogos até uma rodada."""
    # Força FM26
    forcas_fm = {}
    for time in times:
        df_time = df_fm[df_fm['Time'] == time].copy()
        if len(df_time) == 0:
            forcas_fm[time] = 1.0
            continue
        col_ca = [c for c in df_fm.columns
                  if 'CA' in str(c) or 'Rating' in str(c)][0]
        top16 = df_time[col_ca].nlargest(16).mean()
        forcas_fm[time] = top16 ** 3
    media_fm = np.mean(list(forcas_fm.values()))
    forcas_fm = {t: v / media_fm for t, v in forcas_fm.items()}

    # Pontos reais
    pontos = {t: {'P': 0, 'J': 0, 'GP': 0, 'GC': 0} for t in times}
    for _, row in df_jogos_ate_rodada.iterrows():
        m  = row['mandante']
        v  = row['visitante']
        gm = int(row['gols_mandante'])
        gv = int(row['gols_visitante'])
        pontos[m]['J'] += 1; pontos[v]['J'] += 1
        pontos[m]['GP'] += gm; pontos[m]['GC'] += gv
        pontos[v]['GP'] += gv; pontos[v]['GC'] += gm
        if gm > gv:
            pontos[m]['P'] += 3
        elif gv > gm:
            pontos[v]['P'] += 3
        else:
            pontos[m]['P'] += 1; pontos[v]['P'] += 1

    # Força real: aproveitamento (60%) + SG normalizado (40%)
    forcas_reais = {}
    for time in times:
        s = pontos[time]
        aprov = s['P'] / (s['J'] * 3) if s['J'] > 0 else 0.0
        sg    = s['GP'] - s['GC']
        forcas_reais[time] = aprov * 0.6 + sg * 0.4

    min_r = min(forcas_reais.values())
    max_r = max(forcas_reais.values())
    if max_r > min_r:
        forcas_reais = {t: (v - min_r) / (max_r - min_r) + 0.5
                        for t, v in forcas_reais.items()}
    else:
        forcas_reais = {t: 1.0 for t in times}

    media_r = np.mean(list(forcas_reais.values()))
    forcas_reais = {t: v / media_r for t, v in forcas_reais.items()}

    # Híbrida 50/50
    return {t: (forcas_fm[t] + forcas_reais[t]) / 2 for t in times}


def prob_dixon_coles(lambda_m, lambda_v, rho=RHO, n=N_GOLS):
    """
    Calcula P(vitória mandante), P(empate), P(vitória visitante)
    com a correção de Dixon-Coles aplicada nos placares baixos.
    """
    pv, pe, pd = 0.0, 0.0, 0.0
    for g_m in range(n + 1):
        for g_v in range(n + 1):
            prob = (poisson.pmf(g_m, lambda_m) *
                    poisson.pmf(g_v, lambda_v) *
                    tau(g_m, g_v, lambda_m, lambda_v, rho))
            if g_m > g_v:    pv += prob
            elif g_m == g_v: pe += prob
            else:            pd += prob
    total = pv + pe + pd
    return pv / total, pe / total, pd / total


# ── Validação rodada a rodada ─────────────────────────────────────────
brier_scores  = []
brier_poisson = []   # para comparar com o Poisson sem correção
acertos       = []
rodadas_unicas = sorted(df_jogos['rodada'].unique())

print("=" * 75)
print(f"{'VALIDAÇÃO DIXON-COLES — SÉRIE A 2026':^75}")
print(f"{'(ρ = ' + str(RHO) + ')':^75}")
print("=" * 75)
print(f"{'Rodada':<9}{'Mandante':<22}{'Visitante':<22}"
      f"{'Result':>7}{'Previu':>7}{'BS-DC':>8}{'BS-Poi':>8}")
print("-" * 75)

for rodada in rodadas_unicas:
    df_antes    = df_jogos[df_jogos['rodada'] < rodada].copy()
    forcas      = calcular_forcas(df_antes, df_fm, TIMES_SERIE_A)
    jogos_rodada = df_jogos[df_jogos['rodada'] == rodada]

    for _, jogo in jogos_rodada.iterrows():
        m  = jogo['mandante']
        v  = jogo['visitante']
        gm = int(jogo['gols_mandante'])
        gv = int(jogo['gols_visitante'])

        fm = forcas[m]
        fv = forcas[v]

        lambda_m = max(CLIP_MIN, min(MU * fm * VANTAGEM_MANDANTE, CLIP_MAX))
        lambda_v = max(CLIP_MIN, min(MU * fv, CLIP_MAX))

        # Probabilidades com Dixon-Coles
        pv_dc, pe_dc, pd_dc = prob_dixon_coles(lambda_m, lambda_v)

        # Probabilidades Poisson puro (para comparação)
        pv_poi, pe_poi, pd_poi = prob_dixon_coles(lambda_m, lambda_v, rho=0.0)

        if gm > gv:
            r_v, r_e, r_d = 1, 0, 0
            resultado_str = "Casa"
        elif gm == gv:
            r_v, r_e, r_d = 0, 1, 0
            resultado_str = "Emp"
        else:
            r_v, r_e, r_d = 0, 0, 1
            resultado_str = "Fora"

        previsao_str = ("Casa" if pv_dc >= max(pe_dc, pd_dc)
                        else ("Emp" if pe_dc >= pd_dc else "Fora"))

        bs_dc  = (pv_dc  - r_v)**2 + (pe_dc  - r_e)**2 + (pd_dc  - r_d)**2
        bs_poi = (pv_poi - r_v)**2 + (pe_poi - r_e)**2 + (pd_poi - r_d)**2

        brier_scores.append(bs_dc)
        brier_poisson.append(bs_poi)
        acertos.append(1 if resultado_str == previsao_str else 0)

        print(f"  R{int(rodada):<7}{m:<22}{v:<22}"
              f"{resultado_str:>7}{previsao_str:>7}"
              f"{bs_dc:>8.4f}{bs_poi:>8.4f}")

# ── Resumo ────────────────────────────────────────────────────────────
bs_medio_dc  = np.mean(brier_scores)
bs_medio_poi = np.mean(brier_poisson)
acuracia     = np.mean(acertos) * 100
n_jogos      = len(brier_scores)
melhora      = (bs_medio_poi - bs_medio_dc) / bs_medio_poi * 100

print("=" * 75)
print(f"\n{'RESUMO DA VALIDAÇÃO':^75}")
print("=" * 75)
print(f"\n  Jogos validados       : {n_jogos}")
print(f"\n  Brier Score (Dixon-Coles) : {bs_medio_dc:.4f}  "
      f"{'Bom' if bs_medio_dc < 0.25 else 'Acima do ideal'}")
print(f"  Brier Score (Poisson puro): {bs_medio_poi:.4f}")
print(f"  Melhora com Dixon-Coles   : {melhora:.1f}%")
print(f"\n  Acurácia (Dixon-Coles)    : {acuracia:.1f}%")
print(f"\n  Parâmetro ρ (rho)         : {RHO}")
print(f"  (negativo = mais empates; típico para futebol: -0.10 a -0.15)")
print(f"\n  Referência:")
print(f"    BS entre 0.18-0.23 → modelo bem calibrado")
print(f"    BS = 0.25           → equivalente a chute aleatório")
print(f"    BS > 0.25           → pior que aleatório")
print(f"    Modelos profissionais de futebol: BS entre 0.20 e 0.23")

In [ ]:
# =====================================================================
# CÉLULA 10 — TABELA DE VALIDAÇÃO POR JOGO (EXCEL ESTILIZADO)
# =====================================================================
# Exporta uma planilha com:
#   Rodada | Mandante | Visitante | P(Casa) | P(Emp) | P(Fora)
#   | Resultado Real | Previsão | Acerto | Brier Score
#
# Requer que a Célula 9 (Dixon-Coles) já tenha rodado, pois usa
# as funções calcular_forcas() e prob_dixon_coles() definidas lá.
# =====================================================================

import xlsxwriter
from google.colab import files

ARQUIVO = f'Validacao_SerieA_2026_Rodada{RODADA_ATUAL}.xlsx'

wb  = xlsxwriter.Workbook(ARQUIVO)
ws  = wb.add_worksheet('Validação por Jogo')

# ── Formatos ─────────────────────────────────────────────────────────
fmt_titulo = wb.add_format({
    'bold': True, 'font_size': 13, 'font_color': '#FFFFFF',
    'bg_color': '#112233', 'align': 'center', 'valign': 'vcenter',
    'border': 0
})
fmt_subtitulo = wb.add_format({
    'italic': True, 'font_size': 10, 'font_color': '#888888',
    'align': 'center', 'valign': 'vcenter'
})
fmt_header = wb.add_format({
    'bold': True, 'font_color': '#FFFFFF', 'bg_color': '#112233',
    'align': 'center', 'valign': 'vcenter', 'border': 1,
    'text_wrap': True
})
fmt_num = wb.add_format({
    'align': 'center', 'valign': 'vcenter', 'border': 1
})
fmt_pct = wb.add_format({
    'num_format': '0.0%', 'align': 'center',
    'valign': 'vcenter', 'border': 1
})
fmt_bs = wb.add_format({
    'num_format': '0.0000', 'align': 'center',
    'valign': 'vcenter', 'border': 1
})
fmt_time_esq = wb.add_format({
    'align': 'left', 'valign': 'vcenter', 'border': 1, 'indent': 1
})

# Cores por resultado
def fmt_resultado(resultado):
    cores = {
        'Casa':  ('#D5F5E3', '#1E8449'),
        'Emp':   ('#FEF9E7', '#B7950B'),
        'Fora':  ('#FADBD8', '#C0392B'),
    }
    bg, fg = cores.get(resultado, ('#F8F9FA', '#555555'))
    return wb.add_format({
        'bold': True, 'bg_color': bg, 'font_color': fg,
        'align': 'center', 'valign': 'vcenter', 'border': 1
    })

fmt_acerto_sim = wb.add_format({
    'bold': True, 'bg_color': '#D5F5E3', 'font_color': '#1E8449',
    'align': 'center', 'valign': 'vcenter', 'border': 1
})
fmt_acerto_nao = wb.add_format({
    'bg_color': '#FADBD8', 'font_color': '#C0392B',
    'align': 'center', 'valign': 'vcenter', 'border': 1
})

# ── Cabeçalho ─────────────────────────────────────────────────────────
ws.merge_range('A1:K1',
    f'VALIDAÇÃO POR JOGO — BRASILEIRÃO SÉRIE A 2026 — ATÉ RODADA {RODADA_ATUAL}',
    fmt_titulo)
ws.merge_range('A2:K2',
    f'Dixon-Coles (ρ = {RHO}) | MU = {MU} | {len(brier_scores)} jogos validados | '
    f'BS médio = {np.mean(brier_scores):.4f} | Acurácia = {np.mean(acertos)*100:.1f}%',
    fmt_subtitulo)
ws.set_row(0, 28)
ws.set_row(1, 16)
ws.set_row(2, 6)

headers = [
    'Rod.', 'Mandante', 'Visitante',
    'P(Casa)', 'P(Emp)', 'P(Fora)',
    'Resultado', 'Previsão', 'Acerto',
    'Brier Score', 'Δ Maior Prob'
]
for c, h in enumerate(headers):
    ws.write(3, c, h, fmt_header)
ws.set_row(3, 28)

# ── Dados ─────────────────────────────────────────────────────────────
rodadas_unicas = sorted(df_jogos['rodada'].unique())
linha = 4
total_acertos = 0
total_jogos   = 0

for rodada in rodadas_unicas:
    df_antes     = df_jogos[df_jogos['rodada'] < rodada].copy()
    forcas       = calcular_forcas(df_antes, df_fm, TIMES_SERIE_A)
    jogos_rodada = df_jogos[df_jogos['rodada'] == rodada]

    for _, jogo in jogos_rodada.iterrows():
        m  = jogo['mandante']
        v  = jogo['visitante']
        gm = int(jogo['gols_mandante'])
        gv = int(jogo['gols_visitante'])

        fm = forcas[m]
        fv = forcas[v]
        lambda_m = max(CLIP_MIN, min(MU * fm * VANTAGEM_MANDANTE, CLIP_MAX))
        lambda_v = max(CLIP_MIN, min(MU * fv, CLIP_MAX))

        pv, pe, pd = prob_dixon_coles(lambda_m, lambda_v)

        if gm > gv:
            resultado = 'Casa'
            r_v, r_e, r_d = 1, 0, 0
        elif gm == gv:
            resultado = 'Emp'
            r_v, r_e, r_d = 0, 1, 0
        else:
            resultado = 'Fora'
            r_v, r_e, r_d = 0, 0, 1

        previsao = ('Casa' if pv >= max(pe, pd)
                    else ('Emp' if pe >= pd else 'Fora'))
        acerto   = resultado == previsao
        bs       = (pv-r_v)**2 + (pe-r_e)**2 + (pd-r_d)**2

        # Diferença entre a maior probabilidade e a segunda maior
        probs_sorted = sorted([pv, pe, pd], reverse=True)
        delta = probs_sorted[0] - probs_sorted[1]

        # Linha alternada
        bg_alt = '#F5F5F5' if linha % 2 == 0 else '#FFFFFF'
        fmt_alt = wb.add_format({
            'align': 'center', 'valign': 'vcenter',
            'border': 1, 'bg_color': bg_alt
        })
        fmt_alt_esq = wb.add_format({
            'align': 'left', 'valign': 'vcenter',
            'border': 1, 'bg_color': bg_alt, 'indent': 1
        })
        fmt_alt_pct = wb.add_format({
            'num_format': '0.0%', 'align': 'center',
            'valign': 'vcenter', 'border': 1, 'bg_color': bg_alt
        })
        fmt_alt_bs = wb.add_format({
            'num_format': '0.0000', 'align': 'center',
            'valign': 'vcenter', 'border': 1, 'bg_color': bg_alt
        })

        ws.write(linha, 0,  int(rodada),   fmt_alt)
        ws.write(linha, 1,  m,             fmt_alt_esq)
        ws.write(linha, 2,  v,             fmt_alt_esq)
        ws.write(linha, 3,  pv,            fmt_alt_pct)
        ws.write(linha, 4,  pe,            fmt_alt_pct)
        ws.write(linha, 5,  pd,            fmt_alt_pct)
        ws.write(linha, 6,  resultado,     fmt_resultado(resultado))
        ws.write(linha, 7,  previsao,      fmt_resultado(previsao))
        ws.write(linha, 8,  '✅' if acerto else '❌', fmt_acerto_sim if acerto else fmt_acerto_nao)
        ws.write(linha, 9,  bs,            fmt_alt_bs)
        ws.write(linha, 10, delta,         fmt_alt_pct)

        ws.set_row(linha, 18)
        total_acertos += int(acerto)
        total_jogos   += 1
        linha += 1

# ── Linha de totais ───────────────────────────────────────────────────
fmt_total = wb.add_format({
    'bold': True, 'bg_color': '#112233', 'font_color': '#FFFFFF',
    'align': 'center', 'valign': 'vcenter', 'border': 1
})
fmt_total_esq = wb.add_format({
    'bold': True, 'bg_color': '#112233', 'font_color': '#FFFFFF',
    'align': 'left', 'valign': 'vcenter', 'border': 1, 'indent': 1
})

ws.write(linha, 0, '',           fmt_total)
ws.write(linha, 1, 'TOTAL',      fmt_total_esq)
ws.write(linha, 2, f'{total_jogos} jogos', fmt_total_esq)
ws.write(linha, 3, '',           fmt_total)
ws.write(linha, 4, '',           fmt_total)
ws.write(linha, 5, '',           fmt_total)
ws.write(linha, 6, '',           fmt_total)
ws.write(linha, 7, '',           fmt_total)
ws.write(linha, 8, f'{total_acertos}/{total_jogos} ({total_acertos/total_jogos*100:.1f}%)', fmt_total)
ws.write(linha, 9, np.mean(brier_scores), wb.add_format({
    'bold': True, 'num_format': '0.0000', 'bg_color': '#112233',
    'font_color': '#FFFFFF', 'align': 'center', 'valign': 'vcenter', 'border': 1
}))
ws.write(linha, 10, '', fmt_total)
ws.set_row(linha, 22)

# ── Formatação de colunas ─────────────────────────────────────────────
ws.set_column('A:A', 5)    # Rod.
ws.set_column('B:C', 24)   # Times
ws.set_column('D:F', 9)    # Probabilidades
ws.set_column('G:H', 10)   # Resultado / Previsão
ws.set_column('I:I', 8)    # Acerto
ws.set_column('J:J', 11)   # Brier Score
ws.set_column('K:K', 11)   # Delta
ws.freeze_panes(4, 3)

# ── Formatação condicional no Brier Score ─────────────────────────────
# Verde = BS baixo (bom), Vermelho = BS alto (ruim)
ws.conditional_format(4, 9, linha - 1, 9, {
    'type': '3_color_scale',
    'min_color': '#63BE7B',   # verde
    'mid_color': '#FFEB84',   # amarelo
    'max_color': '#F8696B',   # vermelho
    'min_type': 'num', 'min_value': 0,
    'mid_type': 'percentile', 'mid_value': 50,
    'max_type': 'num', 'max_value': 2,
})

wb.close()
files.download(ARQUIVO)
print(f'\n{ARQUIVO} gerado com {total_jogos} jogos!')
print(f'   Acurácia : {total_acertos/total_jogos*100:.1f}%')
print(f'   BS médio : {np.mean(brier_scores):.4f}')